# nb_09_gold_claims_summary

**Layer**: Gold | **Source**: `lh_silver_insurance` | **Target**: `lh_gold_insurance.gold_claims_summary`

**Purpose**: Aggregate claims data by status, type, and year to provide a business-ready claims overview.

**Output Columns**:
| Column | Description |
|---|---|
| claim_year | Year the claim was filed |
| claim_type | Type of claim (collision, fire, etc.) |
| claim_status | Current status (open, approved, etc.) |
| claim_count | Number of claims |
| total_estimated_amount | Sum of estimated amounts |
| avg_estimated_amount | Average estimated amount |
| total_paid_amount | Sum of actual payments made |
| avg_days_to_file | Average days between loss and filing |

**Idempotency**: Uses `overwrite` mode.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, year, count, sum as _sum, avg, round as _round,
    datediff, current_timestamp, coalesce, lit
)

try:
    spark
except NameError:
    spark = SparkSession.builder.appName("nb_09_gold_claims_summary").getOrCreate()

print("nb_09_gold_claims_summary - Gold Layer Aggregation")

In [ ]:
# ============================================================
# Step 1: Read Silver tables
# ============================================================
df_claims = spark.table("claims")
df_payments = spark.table("claim_payments")

print(f"Silver claims:         {df_claims.count():,} rows")
print(f"Silver claim_payments: {df_payments.count():,} rows")

In [ ]:
# ============================================================
# Step 2: Join claims with aggregated payments
# ============================================================
# Aggregate payments per claim
df_payments_agg = df_payments.groupBy("claim_id").agg(
    _sum("payment_amount").alias("total_paid_amount"),
    count("payment_id").alias("payment_count")
)

# Join claims with payment totals
df_enriched = df_claims \
    .withColumn("claim_year", year(col("date_filed"))) \
    .withColumn("days_to_file", datediff(col("date_filed"), col("date_of_loss"))) \
    .join(df_payments_agg, "claim_id", "left") \
    .withColumn("total_paid_amount", coalesce(col("total_paid_amount"), lit(0.0)))

print(f"Enriched claims: {df_enriched.count():,} rows")

In [ ]:
# ============================================================
# Step 3: Aggregate by year, type, status
# ============================================================
df_gold = df_enriched.groupBy("claim_year", "claim_type", "claim_status").agg(
    count("claim_id").alias("claim_count"),
    _round(_sum("estimated_amount"), 2).alias("total_estimated_amount"),
    _round(avg("estimated_amount"), 2).alias("avg_estimated_amount"),
    _round(_sum("total_paid_amount"), 2).alias("total_paid_amount"),
    _round(avg("days_to_file"), 1).alias("avg_days_to_file")
).orderBy("claim_year", "claim_type", "claim_status")

gold_count = df_gold.count()
print(f"Gold claims summary: {gold_count:,} aggregated rows")
df_gold.show(10, truncate=False)

In [ ]:
# ============================================================
# Step 4: Write to Gold
# ============================================================
df_gold.withColumn("_processed_at", current_timestamp()) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_claims_summary")

print(f"✓ {gold_count:,} rows written to gold_claims_summary")

In [ ]:
# ============================================================
# Validation
# ============================================================
readback = spark.table("gold_claims_summary").count()
assert readback == gold_count, f"Mismatch: {readback} vs {gold_count}"
print(f"✓ Verification passed: {readback:,} rows in gold_claims_summary")

# Quick KPI check
print("\n--- Claims Summary KPIs ---")
spark.sql("""
    SELECT 
        SUM(claim_count) as total_claims,
        ROUND(SUM(total_estimated_amount), 2) as total_estimated,
        ROUND(SUM(total_paid_amount), 2) as total_paid,
        ROUND(SUM(total_paid_amount) / NULLIF(SUM(total_estimated_amount), 0) * 100, 1) as payout_ratio_pct
    FROM gold_claims_summary
""").show(truncate=False)